# Notebook 1 · Physics simulation: a projectile with air resistance

**Goal:** build a computer simulation of a projectile (like a ball being thrown) that includes air resistance. We'll use it in Notebook 2 to create data for machine learning.

**A-level links**
- **Physics:** Newton's 2nd law, SUVAT, resolving vectors, drag forces
- **Further Maths:** differential equations, numerical methods (Euler's method), errors and order of convergence

Run each cell in order with **Shift + Enter**.

## 1. The physics

### Without air resistance (the A-level model)
The only force is gravity, so the acceleration is $(0, -g)$. SUVAT gives

$$x = v_0\cos\theta\, t, \qquad y = v_0\sin\theta\, t - \tfrac{1}{2}g t^2$$

and from these you can derive:

| Quantity | Formula |
|---|---|
| Range | $R = \dfrac{v_0^2 \sin 2\theta}{g}$ |
| Time of flight | $T = \dfrac{2 v_0 \sin\theta}{g}$ |
| Maximum height | $H = \dfrac{v_0^2 \sin^2\theta}{2g}$ |

So the best angle for maximum range is always $45°$.

### With air resistance
At normal speeds, drag is roughly proportional to **speed squared**, and it acts opposite to the velocity:

$$\mathbf{F}_{\text{drag}} = -k\,|\mathbf{v}|\,\mathbf{v}, \qquad k = \tfrac{1}{2}\rho C_d A$$

($\rho$ = air density, $C_d$ = drag coefficient, $A$ = cross-sectional area.)

Newton's 2nd law, $m\mathbf{a} = m\mathbf{g} + \mathbf{F}_{\text{drag}}$, split into components gives:

$$\frac{dv_x}{dt} = -b\,v\,v_x, \qquad \frac{dv_y}{dt} = -g - b\,v\,v_y, \qquad v = \sqrt{v_x^2 + v_y^2}, \qquad b = \frac{k}{m}$$

These are **coupled, non-linear differential equations**. Because $v$ depends on both $v_x$ and $v_y$, there is no neat formula for the answer. Instead we solve them **numerically**, which is exactly what the numerical methods topic in Further Maths is about.

### Some real values of $b = k/m$
| Ball | $b$ (m⁻¹) |
|---|---|
| Golf ball | ≈ 0.005 |
| Cricket ball | ≈ 0.006 |
| Football | ≈ 0.013 |
| Tennis ball | ≈ 0.020 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

g = 9.81  # gravitational field strength, m/s^2

## 2. Writing the problem as a "state vector"

We group everything we need to know at one instant into a **state** $\mathbf{s} = (x, y, v_x, v_y)$.
Then the whole problem becomes one first-order differential equation:

$$\frac{d\mathbf{s}}{dt} = \mathbf{f}(\mathbf{s}) = \big(v_x,\; v_y,\; -b v v_x,\; -g - b v v_y\big)$$

The function below calculates $\mathbf{f}(\mathbf{s})$.

In [ ]:
def derivatives(state, b):
    """Return ds/dt for the state s = (x, y, vx, vy)."""
    x, y, vx, vy = state
    v = np.sqrt(vx**2 + vy**2)
    return np.array([vx, vy, -b * v * vx, -g - b * v * vy])

## 3. Euler's method

Euler's method takes a small time step $h$ and assumes the gradient stays constant during the step (it follows the tangent line):

$$\mathbf{s}_{n+1} = \mathbf{s}_n + h\,\mathbf{f}(\mathbf{s}_n)$$

It is simple, but the error is proportional to $h$ (a **first-order** method). Halving the step size only halves the error.

In [ ]:
def euler_step(state, dt, b):
    return state + dt * derivatives(state, b)

## 4. The Runge–Kutta method (RK4)

RK4 measures the gradient **four times** in each step (at the start, twice in the middle and at the end) and takes a weighted average. It works a bit like Simpson's rule:

$$\mathbf{k}_1 = \mathbf{f}(\mathbf{s}_n), \quad \mathbf{k}_2 = \mathbf{f}\!\left(\mathbf{s}_n + \tfrac{h}{2}\mathbf{k}_1\right), \quad \mathbf{k}_3 = \mathbf{f}\!\left(\mathbf{s}_n + \tfrac{h}{2}\mathbf{k}_2\right), \quad \mathbf{k}_4 = \mathbf{f}(\mathbf{s}_n + h\mathbf{k}_3)$$

$$\mathbf{s}_{n+1} = \mathbf{s}_n + \frac{h}{6}\left(\mathbf{k}_1 + 2\mathbf{k}_2 + 2\mathbf{k}_3 + \mathbf{k}_4\right)$$

Its error is proportional to $h^4$ (**fourth order**), so halving the step size makes the error about **16 times** smaller. We'll test that claim in Section 8.

In [ ]:
def rk4_step(state, dt, b):
    k1 = derivatives(state, b)
    k2 = derivatives(state + 0.5 * dt * k1, b)
    k3 = derivatives(state + 0.5 * dt * k2, b)
    k4 = derivatives(state + dt * k3, b)
    return state + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

## 5. Simulating a whole flight

Keep taking steps until the projectile goes below the ground ($y < 0$). The landing happens partway through the last step, so we use **linear interpolation** between the last two points to find where $y = 0$.

In [ ]:
def simulate(v0, angle_deg, b, dt=0.01, step=rk4_step):
    """Launch from (0, 0) and step forward until landing.

    Returns (times, states), where states has one row (x, y, vx, vy) per time.
    """
    theta = np.radians(angle_deg)
    state = np.array([0.0, 0.0, v0 * np.cos(theta), v0 * np.sin(theta)])
    t = 0.0
    times, states = [t], [state]

    while True:
        new_state = step(state, dt, b)
        if new_state[1] < 0:
            # Linear interpolation to find where y = 0 during this step
            fraction = state[1] / (state[1] - new_state[1])
            landing = state + fraction * (new_state - state)
            landing[1] = 0.0
            times.append(t + fraction * dt)
            states.append(landing)
            break
        state = new_state
        t += dt
        times.append(t)
        states.append(state)

    return np.array(times), np.array(states)

## 6. Test: does the simulation agree with SUVAT?

Before trusting a simulation, check it against a case where we already know the answer. With $b = 0$ (no air resistance), it should match the SUVAT formulas.

In [ ]:
v0, angle = 20.0, 40.0
t, s = simulate(v0, angle, b=0.0)

theta = np.radians(angle)
R_exact = v0**2 * np.sin(2 * theta) / g
T_exact = 2 * v0 * np.sin(theta) / g
H_exact = (v0 * np.sin(theta))**2 / (2 * g)

print(f"Range:          simulated {s[-1, 0]:8.4f} m   exact {R_exact:8.4f} m")
print(f"Time of flight: simulated {t[-1]:8.4f} s   exact {T_exact:8.4f} s")
print(f"Max height:     simulated {s[:, 1].max():8.4f} m   exact {H_exact:8.4f} m")

In [ ]:
t_exact = np.linspace(0, T_exact, 200)
x_exact = v0 * np.cos(theta) * t_exact
y_exact = v0 * np.sin(theta) * t_exact - 0.5 * g * t_exact**2

plt.figure(figsize=(8, 4))
plt.plot(x_exact, y_exact, "k-", lw=3, alpha=0.3, label="SUVAT (exact)")
plt.plot(s[::10, 0], s[::10, 1], "o", ms=4, label="RK4 simulation")
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.title("No air resistance: simulation vs exact answer")
plt.legend()
plt.axis("equal")
plt.show()

## 7. Adding air resistance

Now we turn on drag. Compare the same launch ($30$ m/s at $45°$) for different values of $b$.

In [ ]:
plt.figure(figsize=(9, 4.5))
for b in [0, 0.005, 0.01, 0.02]:
    t, s = simulate(30, 45, b)
    plt.plot(s[:, 0], s[:, 1], label=f"b = {b}:  range = {s[-1, 0]:.1f} m")
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.title("Launch at 30 m/s, 45°")
plt.legend()
plt.axis("equal")
plt.show()

**What to notice:**
- Drag reduces the range a lot. A tennis ball ($b \approx 0.02$) travels less than half as far as it would in a vacuum.
- The path is **no longer symmetric**. The projectile comes down more steeply than it went up, because it has lost horizontal speed to drag.

## 8. How accurate are Euler and RK4? (Further Maths)

If a method's error is $E = C h^p$, then taking logs gives

$$\log E = \log C + p \log h,$$

which is a **straight line** with gradient $p$ (the *order* of the method). We can measure $p$ by:
1. computing a very accurate "reference" answer with a tiny step size,
2. running each method with several step sizes $h$ and measuring the error at $t = 2$ s,
3. plotting $\log E$ against $\log h$ and finding the gradient.

In [ ]:
def state_at_time(v0, angle_deg, b, t_end, dt, step):
    """Step forward for a fixed time (ignoring the ground)."""
    theta = np.radians(angle_deg)
    state = np.array([0.0, 0.0, v0 * np.cos(theta), v0 * np.sin(theta)])
    for _ in range(round(t_end / dt)):
        state = step(state, dt, b)
    return state

v0, angle, b, t_end = 30.0, 45.0, 0.01, 2.0
reference = state_at_time(v0, angle, b, t_end, dt=1e-4, step=rk4_step)

step_sizes = np.array([0.2, 0.1, 0.05, 0.02, 0.01, 0.005])
errors = {"Euler": [], "RK4": []}
for h in step_sizes:
    for name, step in [("Euler", euler_step), ("RK4", rk4_step)]:
        s_end = state_at_time(v0, angle, b, t_end, h, step)
        errors[name].append(np.linalg.norm(s_end[:2] - reference[:2]))  # position error in m

plt.figure(figsize=(7, 4.5))
for name, errs in errors.items():
    gradient, _ = np.polyfit(np.log10(step_sizes), np.log10(errs), 1)
    plt.loglog(step_sizes, errs, "o-", label=f"{name}: gradient = {gradient:.2f}")
plt.xlabel("step size h (s)")
plt.ylabel("position error after 2 s (m)")
plt.title("Error vs step size (log-log)")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

**Result:** the gradients should come out close to **1 for Euler** and **4 for RK4**, which matches the theory.

With $h = 0.01$ s, RK4's error is far smaller than a millimetre, so we'll use **RK4 with `dt = 0.01`** to generate our data.

➡️ The functions from this notebook are also saved in `projectile.py`, so the next notebooks can simply `import` them.

**Next:** open `02_generate_dataset.ipynb`.